# Colab-ready 教學入口：Ch01 從資料中學習的機器

本 notebook 由官方程式碼 notebook 產生，第一格加入 Colab setup，讓學生不需要手動 clone repo 或切換工作目錄。

- 官方來源：`ch01/ch01.ipynb`
- 官方 repo：https://github.com/rasbt/python-machine-learning-book-3rd-edition.git
- 書籍：Sebastian Raschka and Vahid Mirjalili, *Python Machine Learning, 3rd Ed.*, Packt Publishing, 2019
- 程式碼授權：MIT License，請參考本 repo 的 `THIRD_PARTY_NOTICES.md`

上課時請先執行下一格 setup，再依序執行原 notebook。若深度學習或大型資料章節耗時過久，請改用課堂 quick mode 或由講師示範重點 cell。


In [ ]:
# @title Colab setup for Python Machine Learning 3rd ed.
import importlib
import os
import platform
import subprocess
import sys

REPO_URL = "https://github.com/rasbt/python-machine-learning-book-3rd-edition.git"
REPO_DIR = "/content/python-machine-learning-book-3rd-edition" if os.path.exists("/content") else os.path.abspath("_python_ml_3e_official")
CHAPTER_DIR = "ch01"
EXTRA_PACKAGES = ['watermark', 'mlxtend', 'pyprind']

def _run(cmd):
    print("$", " ".join(cmd))
    subprocess.run(cmd, check=True)

if not os.path.isdir(REPO_DIR):
    _run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR])

target_dir = os.path.join(REPO_DIR, CHAPTER_DIR)
os.chdir(target_dir)
print("Working directory:", os.getcwd())

def ensure_import(import_name, pip_name=None):
    pip_name = pip_name or import_name
    if importlib.util.find_spec(import_name) is None:
        _run([sys.executable, "-m", "pip", "install", "-q", pip_name])

for import_name, pip_name in [
    ("numpy", "numpy"),
    ("pandas", "pandas"),
    ("matplotlib", "matplotlib"),
    ("sklearn", "scikit-learn"),
    ("scipy", "scipy"),
]:
    ensure_import(import_name, pip_name)

for pkg in EXTRA_PACKAGES:
    import_name = pkg.split("==")[0].split("[")[0].replace("-", "_")
    if pkg.startswith("tensorflow-datasets"):
        import_name = "tensorflow_datasets"
    if pkg.startswith("scikit-learn"):
        import_name = "sklearn"
    if pkg.startswith("gym=="):
        import_name = "gym"
    ensure_import(import_name, pkg)

import numpy as np

# Compatibility shims for the current Colab runtime family.
# Colab 2026.04 lists Python 3.12.13, NumPy 2.0.2, and TensorFlow 2.19.0.
# The 2019 book notebooks still use a few aliases/API shapes from older releases.
if not hasattr(np, "float"):
    np.float = float
if not hasattr(np, "int"):
    np.int = int
if not hasattr(np, "bool8"):
    np.bool8 = np.bool_

try:
    import matplotlib
    import matplotlib.pyplot as plt
    matplotlib.rcParams["figure.figsize"] = (7, 5)
except Exception as exc:
    print("matplotlib setup skipped:", type(exc).__name__, exc)

try:
    import tensorflow as tf
    print("TensorFlow devices:", [device.device_type + ":" + device.name.split(":")[-1] for device in tf.config.list_physical_devices()])
except Exception as exc:
    print("TensorFlow not loaded:", type(exc).__name__, exc)

try:
    import gym

    if not getattr(gym, "_pyml3e_old_api_patch", False):
        _original_gym_make = gym.make

        class _OldStepAPIWrapper(gym.Wrapper):
            def reset(self, *args, **kwargs):
                result = self.env.reset(*args, **kwargs)
                if isinstance(result, tuple) and len(result) == 2:
                    return result[0]
                return result

            def step(self, action):
                result = self.env.step(action)
                if isinstance(result, tuple) and len(result) == 5:
                    obs, reward, terminated, truncated, info = result
                    return obs, reward, bool(terminated or truncated), info
                return result

        def _patched_make(*args, **kwargs):
            env = _original_gym_make(*args, **kwargs)
            return _OldStepAPIWrapper(env)

        gym.make = _patched_make
        gym._pyml3e_old_api_patch = True
        print("Gym old-step API wrapper enabled.")
except Exception as exc:
    print("Gym compatibility setup skipped:", type(exc).__name__, exc)

print("Python:", sys.version.split()[0], "| Platform:", platform.platform())
for mod_name in ["numpy", "pandas", "matplotlib", "sklearn", "tensorflow", "tensorflow_datasets", "gym"]:
    try:
        mod = importlib.import_module(mod_name)
        print(f"{mod_name}:", getattr(mod, "__version__", "installed"))
    except Exception as exc:
        print(f"{mod_name}: not loaded ({type(exc).__name__})")

print("Setup complete. Run the notebook cells below in order.")


Copyright (c) 2019 [Sebastian Raschka](sebastianraschka.com)

https://github.com/rasbt/python-machine-learning-book-3rd-edition

[MIT License](https://github.com/rasbt/python-machine-learning-book-3rd-edition/blob/master/LICENSE.txt)

# Python Machine Learning - Code Examples

# Chapter 1 - Giving Computers the Ability to Learn from Data

### Overview

- [Building intelligent machines to transform data into knowledge](#Building-intelligent-machines-to-transform-data-into-knowledge)
- [The three different types of machine learning](#The-three-different-types-of-machine-learning)
    - [Making predictions about the future with supervised learning](#Making-predictions-about-the-future-with-supervised-learning)
        - [Classification for predicting class labels](#Classification-for-predicting-class-labels)
        - [Regression for predicting continuous outcomes](#Regression-for-predicting-continuous-outcomes)
    - [Solving interactive problems with reinforcement learning](#Solving-interactive-problems-with-reinforcement-learning)
    - [Discovering hidden structures with unsupervised learning](#Discovering-hidden-structures-with-unsupervised-learning)
        - [Finding subgroups with clustering](#Finding-subgroups-with-clustering)
        - [Dimensionality reduction for data compression](#Dimensionality-reduction-for-data-compression)
        - [An introduction to the basic terminology and notations](#An-introduction-to-the-basic-terminology-and-notations)
- [A roadmap for building machine learning systems](#A-roadmap-for-building-machine-learning-systems)
    - [Preprocessing - getting data into shape](#Preprocessing--getting-data-into-shape)
    - [Training and selecting a predictive model](#Training-and-selecting-a-predictive-model)
    - [Evaluating models and predicting unseen data instances](#Evaluating-models-and-predicting-unseen-data-instances)
- [Using Python for machine learning](#Using-Python-for-machine-learning)
- [Installing Python packages](#Installing-Python-packages)
- [Summary](#Summary)

<br>
<br>

In [ ]:
from IPython.display import Image


# Building intelligent machines to transform data into knowledge

...

# The three different types of machine learning

In [ ]:
Image(filename='./images/01_01.png', width=500) 


<br>
<br>

## Making predictions about the future with supervised learning

In [ ]:
Image(filename='./images/01_02.png', width=500) 


<br>
<br>

### Classification for predicting class labels

In [ ]:
Image(filename='./images/01_03.png', width=300) 


<br>
<br>

### Regression for predicting continuous outcomes

In [ ]:
Image(filename='./images/01_04.png', width=300) 


<br>
<br>

## Solving interactive problems with reinforcement learning

In [ ]:
Image(filename='./images/01_05.png', width=300) 


<br>
<br>

## Discovering hidden structures with unsupervised learning

...

### Finding subgroups with clustering

In [ ]:
Image(filename='./images/01_06.png', width=300) 


<br>
<br>

### Dimensionality reduction for data compression

In [ ]:
Image(filename='./images/01_07.png', width=500) 


<br>
<br>

### An introduction to the basic terminology and notations

In [ ]:
Image(filename='./images/01_08.png', width=500) 


<br>
<br>

# A roadmap for building machine learning systems

In [ ]:
Image(filename='./images/01_09.png', width=700) 


<br>
<br>

## Preprocessing - getting data into shape

...

## Training and selecting a predictive model

...

## Evaluating models and predicting unseen data instances

...

# Using Python for machine learning

...

# Installing Python packages

...

# Summary

...